# Data Structures — CO1

**2310214L.CO.1** — *Develop applications using arrays.* **[L3]**

---

## The idea

An array stores items back to back, so **position is identity**.

```
address of item i  =  start  +  ( i × size of one item )
```

One multiplication, one addition — the same cost whether the array holds 8 items or 8,000.
A linked list has to walk from the front.

**Our rule:** identified by position → array.

| What | Index means | File |
|---|---|---|
| Register file | the register number | `src/core/RegisterFile.h` |
| Instruction memory | the program counter | `src/core/CPU.cpp` |
| Fetch queue | place in the line | `src/ds/CircularQueue.h` |
| Screen buffer | the pixel's position | `src/ui/Canvas.cpp` |

![the register file as an array](diagrams/ds-arrays.png)

---

## 1 · Registers

```cpp
class RegisterFile {
private:
    Register gp_[NUM_GP_REGISTERS];   // R0..R7
    Register pc_;                     // program counter
    Register ir_;                     // instruction register
    Register sp_;                     // stack pointer
    Register mar_;                    // memory address register
    Register mdr_;                    // memory data register
    Flags    flags_;
};
```
<sub>src/core/RegisterFile.h:49</sub>

```cpp
Word RegisterFile::readGP(int index) {
    if (index < 0 || index >= NUM_GP_REGISTERS)
        throw InvalidRegisterException("read index out of range");
    return gp_[index].read();
}

void RegisterFile::writeGP(int index, Word value) {
    if (index < 0 || index >= NUM_GP_REGISTERS)
        throw InvalidRegisterException("write index out of range");
    gp_[index].write(value);
}
```
<sub>src/core/RegisterFile.cpp:32</sub>

**R5 *is* `gp_[5]`.** No lookup — one step, on every instruction.

---

## 2 · Instruction memory

```cpp
Instruction* CPU::instructionAt(unsigned int address) const {
    if (address < programBase_) return 0;
    unsigned int idx = address - programBase_;
    if (idx >= programSize_) return 0;
    return program_[idx];
}
```
<sub>src/core/CPU.cpp:61</sub>

**The program counter is the index.** A jump doesn't *find* a line — it writes a number
into the PC.

---

## 3 · The circular queue

A plain array queue either shifts everything on removal, or runs off the end. Move the
**index** instead, and wrap it.

```cpp
void enqueue(const T& value) {
    if (count_ == capacity_)
        throw core::IndexOutOfRangeException("CircularQueue::enqueue on full queue");
    buf_[(front_ + count_) % capacity_] = value;
    ++count_;
}

T dequeue() {
    if (count_ == 0)
        throw core::IndexOutOfRangeException("CircularQueue::dequeue on empty queue");
    T value = buf_[front_];
    front_  = (front_ + 1) % capacity_;
    --count_;
    return value;
}

const T& at(size_t i) const {                 // logical index 0 == front
    if (i >= count_) throw core::IndexOutOfRangeException("CircularQueue::at");
    return buf_[(front_ + i) % capacity_];
}
```
<sub>src/ds/CircularQueue.h:52</sub>

```
             0     1     2     3
           [ - ] [ - ] [ A ] [ B ]     front_ = 2, count_ = 2

add C  ->  (2 + 2) % 4 = 0             wraps to the front of the array

             0     1     2     3
           [ C ] [ - ] [ A ] [ B ]     front_ = 2, count_ = 3
```

Nothing copied, nothing reallocated. Real fetch buffers work this way.

---

## 4 · Screen buffer — 2D inside 1D

```cpp
Canvas::Canvas(int width, int height)
    : width_(width > 0 ? width : 1),
      height_(height > 0 ? height : 1),
      buffer_(0) {
    buffer_ = new Pixel[width_ * height_];
    clear();
}

void Canvas::setPixel(int x, int y, Pixel p) {
    if (inBounds(x, y)) buffer_[y * width_ + x] = p;
}

Pixel Canvas::getPixel(int x, int y) const {
    if (!inBounds(x, y)) return PX_EMPTY;
    return buffer_[y * width_ + x];
}
```
<sub>src/ui/Canvas.cpp:6</sub>

```
width = 5                          pixel (x=3, y=2)

row 0:  [ 0] [ 1] [ 2] [ 3] [ 4]   skip 2 rows  ->  2 × 5 = 10
row 1:  [ 5] [ 6] [ 7] [ 8] [ 9]   step 3 along ->  10 + 3
row 2:  [10] [11] [12] [13] [14]                    index 13
```

Every line, circle and fill in the project ends up here.

---

## What an array will not do

It checks nothing — `buffer_[999999]` compiles and reads whatever is there.

```cpp
bool Canvas::inBounds(int x, int y) const {
    return x >= 0 && x < width_ && y >= 0 && y < height_;
}
```
<sub>src/ui/Canvas.cpp:33</sub>

So every index from outside is guarded first.

---

## In one minute

1. **Position is identity** — item 5 by arithmetic, not searching.
2. Four uses: registers by number, instructions by PC, queue slots by order, pixels by
   coordinate.
3. **Circular queue** wraps the *index* with `%` instead of moving the *data*.
4. **Screen buffer** folds 2D into 1D with `y * width + x`.
5. An array trusts you completely, so we do the checking.